In [1]:
from agents import *
from config.agents_io import *

### Communication agent

In [2]:
agent = CommunicationAgent()

In [3]:
input_data = CommunicationInput(
    conversation_history=[

    ],
    user_query="Update the feature engineering logic in my code to perform categorical encoding for categorical data and replace null values with the most repeating feature."
)

# Call the extract_intent method
output = await agent.extract_intent(input_data)

# Print output
print("\n=== OUTPUT OBJECT ===")
print(output)

INFO:httpx:HTTP Request: POST https://api.ai-gateway.tigeranalytics.com/chat/completions "HTTP/1.1 200 OK"



=== OUTPUT OBJECT ===
core_intent='Update feature engineering logic to perform categorical encoding and replace null values with the most frequent feature.' context_notes='The user wants to handle categorical data by encoding it and address missing values by replacing them with the most frequently occurring feature value.' success=True message='Intent extracted successfully'


### Query Enhancer Agent

In [4]:
query_rephrase = QueryRephraserAgent()

In [5]:
# 4️⃣ Prepare input for query rephraser using comm_output
rephrase_input = QueryEnhancerInput(
core_intent=output.core_intent,
context_notes=output.context_notes
)

# 5️⃣ Second stage: rephrase query
rephrase_output = await query_rephrase.enhance_query(rephrase_input)

INFO:httpx:HTTP Request: POST https://api.ai-gateway.tigeranalytics.com/chat/completions "HTTP/1.1 200 OK"


In [6]:
rephrase_output

QueryEnhancerOutput(developer_task='Update feature engineering logic to perform categorical encoding and replace null values with the most frequent feature value.', is_satisfied=True, suggestions=[], success=True, message='LLM success', reason="The request involves modifying the application's feature engineering logic, which is part of the source code and not a configuration change.", change_type='code_change')

### Document Generator Agent

In [7]:
agent = DocumentGeneratorAgent()

query = rephrase_output.developer_task
input_data = DocumentGeneratorInput(
    developer_task_query = query
)
rag_output = await agent.generate_document(input_data)

The query is  Update feature engineering logic to perform categorical encoding and replace null values with the most frequent feature value.


INFO:httpx:HTTP Request: POST https://api.ai-gateway.tigeranalytics.com/chat/completions "HTTP/1.1 200 OK"
INFO:agents.Document_generation_agent.document_generation_agent:LLM selected tool: hybrid_search
INFO:agents.Document_generation_agent.document_generation_agent:*******************************************************************************
INFO:agents.Document_generation_agent.document_generation_agent:Reasoning: The query is conceptual and semantic in nature, as it refers to 'feature engineering logic' and describes specific operations (categorical encoding and replacing null values). It does not mention specific file names, classes, or functions, ruling out the use of 'identify_file'. Additionally, the query does not ask for specific code patterns or keywords, making 'keyword_match' unsuitable. 'Hybrid_search' is the most appropriate tool because it is designed to handle complex queries requiring semantic understanding, which aligns with the user's request to locate and update 

Selected Tool : hybrid_search
Parameters: {
  "query": "Update feature engineering logic to perform categorical encoding and replace null values with the most frequent feature value.",
  "focus_area": "all"
}
LLM Reasoning: The query is conceptual and semantic in nature, as it refers to 'feature engineering logic' and describes specific operations (categorical encoding and replacing null values). It does not mention specific file names, classes, or functions, ruling out the use of 'identify_file'. Additionally, the query does not ask for specific code patterns or keywords, making 'keyword_match' unsuitable. 'Hybrid_search' is the most appropriate tool because it is designed to handle complex queries requiring semantic understanding, which aligns with the user's request to locate and update logic related to feature engineering.

 Step 2: Executing hybrid_search tool...
******************* Inside Hybrid Search tool *********************
************************ Inside vector search for s

INFO:agents.Document_generation_agent.document_generation_agent:PostgreSQL connection pool initialized


inside get_file_info_for_vector
inside get_class_info_for_vector
inside get_function_info_for_vector
before going to enhanced results 8


INFO:httpx:HTTP Request: POST https://api.ai-gateway.tigeranalytics.com/chat/completions "HTTP/1.1 200 OK"


printing rankings: [{'index': 0, 'relevance': 9.2, 'reason': "Perfect match - The 'FeatureEngineering' class in 'feature_engineering.py' appears to directly address the query by providing feature engineering logic, which is likely to include categorical encoding and handling null values."}, {'index': 2, 'relevance': 8.5, 'reason': "Highly relevant - The 'add_features' function within the 'FeatureEngineering' class in 'feature_engineering.py' is likely a specific implementation of feature engineering logic, making it directly useful for the task."}]
after enhancing 2
COMPLETED EXTRACTING QUERY TERMS


INFO:agents.Document_generation_agent.document_generation_agent:Tool 2 found 1 hybrid matches


examples\processing\feature_engineering.py 0.9199999999999999
examples\processing\feature_engineering.py 0.85
meta_datasearch: examples\processing\feature_engineering.py
meta_datasearch: examples\processing\feature_engineering.py
meta_datasearch: examples\processing\feature_engineering.py
combined results examples\processing\feature_engineering.py


In [8]:
rag_output

DocumentGeneratorOutput(generated_doc='[{\'file_path\': \'examples\\\\processing\\\\feature_engineering.py\', \'name\': \'add_features\', \'type\': \'function\', \'relevance_score\': 1.05, \'enhanced_content\': \'Here\\\'s the function rewritten with a comprehensive docstring and comments that clarify its purpose, parameters, return value, and usage notes:\\n\\n```python\\nfrom pyspark.ml.feature import VectorAssembler\\nfrom pyspark.sql import DataFrame\\n\\ndef add_features(df: DataFrame, feature_cols: list) -> DataFrame:\\n    """\\n    Combines multiple feature columns into a single vector feature column in a Spark DataFrame.\\n\\n    This function utilizes the `VectorAssembler` from PySpark to merge specified feature columns\\n    into a single "features" column, which is required input for many machine learning models in Spark.\\n\\n    Parameters:\\n    ----------\\n    df : pyspark.sql.DataFrame\\n        The Spark DataFrame that contains the input data.\\n        This should i

### Master planner agent

In [9]:
parsed_config = {
    "project_type" : "python",
    "framework" : "Crud operation",
    "migration_target" : "python",
    "preserve_functionality" : True
}

user_question = query
rag_output = rag_output.generated_doc

In [10]:
agent = MasterPlannerAgent()

planner_input = MasterPlannerInput(
    parsed_config = parsed_config,
    user_question = user_question
)

result = await agent.identify_target_files(
    input_data = planner_input,
    rag_result = rag_output
)

INFO:agents.master_planner_agent.master_planner_agent:🔍 Starting RAG-only file identification process...
INFO:agents.master_planner_agent.master_planner_agent:Extracted specific files: []
INFO:agents.master_planner_agent.master_planner_agent:🤖 Processing RAG output for file identification...
INFO:httpx:HTTP Request: POST https://api.ai-gateway.tigeranalytics.com/chat/completions "HTTP/1.1 200 OK"
INFO:agents.master_planner_agent.master_planner_agent:✅ Identified 1 files from RAG analysis
INFO:agents.master_planner_agent.master_planner_agent:📊 Analysis confidence: high
INFO:agents.master_planner_agent.master_planner_agent:📋 Summary: Based on the RAG analysis and user query, the file `examples/processing/feature_engineering.py` requires modifications to implement categorical encoding and handle null values by replacing them with the most frequent feature value. No additional files were identified for modification or creation.
INFO:agents.master_planner_agent.master_planner_agent:✅ Create

In [11]:
result

MasterPlannerOutput(success=True, message='Successfully identified 1 files for modification based on RAG analysis.', files_to_modify=[TargetFileOutput(file_path='examples/processing/feature_engineering.py', file_info={'size': 0, 'exists': True, 'extension': '.py', 'is_python': True}, analysis=FileAnalysisResult(needs_modification=True, modification_type='data_transformation', priority='high', reason='The user requested updates to the feature engineering logic to include categorical encoding and replace null values with the most frequent feature value. This aligns with the function `add_features` identified in the RAG analysis.', cross_file_dependencies={'depends_on': [], 'affects': [], 'imports_from': ['pyspark.ml.feature.VectorAssembler'], 'imported_by': [], 'dependency_reason': "The function relies on PySpark's `VectorAssembler` for feature column aggregation, which may need adjustments to accommodate the new preprocessing logic."}), priority='high')])

In [12]:
def convert_master_planner_to_delta_input(master_result):
    target_files_dict = []

    for target_file_obj in master_result.files_to_modify:
        try:
            file_dict = {
                "file_path" : target_file_obj.file_path,
                "file_info" : target_file_obj.file_info if target_file_obj.file_info else {},
                "analysis" : {
                    "needs_modification" : target_file_obj.analysis.needs_modification,
                    "modification_type" :  target_file_obj.analysis.modification_type or "general",
                    "priority" : target_file_obj.analysis.priority or "medium",
                    "reason" : target_file_obj.analysis.reason or "Modification needed",
                    "cross_file_dependencies" : target_file_obj.analysis.cross_file_dependencies or 'nothing'
                },
                "priority" :  target_file_obj.priority
            }
            target_files_dict.append(file_dict)

        except Exception as e:
            print(f"Error converting file {getattr(target_file_obj, 'file_path', 'unknown.py')} : {e}")
            file_dict = {
                "file_path" : getattr(target_file_obj, 'file_path', 'unknown.py'),
                "file_info" : {},
                "analysis" : {
                    "needs_modification" : True,
                    "modification_type" :  "general",
                    "priority" : "medium",
                    "reason" : "Fallback conversion"
                },
                "priority" :  "medium"
            }
            target_files_dict.append(file_dict)
    return target_files_dict


In [13]:
agent = DeltaAnalyzerAgent()
req = convert_master_planner_to_delta_input(result)

req

[{'file_path': 'examples/processing/feature_engineering.py',
  'file_info': {'size': 0,
   'exists': True,
   'extension': '.py',
   'is_python': True},
  'analysis': {'needs_modification': True,
   'modification_type': 'data_transformation',
   'priority': 'high',
   'reason': 'The user requested updates to the feature engineering logic to include categorical encoding and replace null values with the most frequent feature value. This aligns with the function `add_features` identified in the RAG analysis.',
   'cross_file_dependencies': {'depends_on': [],
    'affects': [],
    'imports_from': ['pyspark.ml.feature.VectorAssembler'],
    'imported_by': [],
    'dependency_reason': "The function relies on PySpark's `VectorAssembler` for feature column aggregation, which may need adjustments to accommodate the new preprocessing logic."}},
  'priority': 'high'}]

In [14]:
result_data = await agent.create_modification_plan(req, parsed_config, user_question)

INFO:agents.delta_analyzer_agent.delta_analyzer_agent:[DeltaAnalyzerAgent] Using filename only: examples/processing/feature_engineering.py


in if


INFO:httpx:HTTP Request: POST https://api.ai-gateway.tigeranalytics.com/chat/completions "HTTP/1.1 200 OK"


In [15]:
result_data

{'files_to_modify': [{'file_path': 'examples/processing/feature_engineering.py',
   'priority': 'high',
   'modification_type': 'data_transformation',
   'suggestions': {'modifications': [{'action': 'modify',
      'target_type': 'function',
      'target_name': 'add_features',
      'line_number': 7,
      'old_code': 'def add_features(df: DataFrame, feature_cols: list):\n    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")\n    df_features = assembler.transform(df)\n    return df_features',
      'new_code': 'def add_features(df: DataFrame, feature_cols: list):\n    from pyspark.sql.functions import col, count, when, lit, mode\n    \n    # Replace null values with the most frequent value for each column\n    for col_name in feature_cols:\n        most_frequent_value = df.groupBy(col_name).agg(count(col_name).alias("count")).orderBy("count", ascending=false).first()[0]\n        df = df.withColumn(col_name, when(col(col_name).isNull(), lit(most_frequent_value))

### Code Generator agent

In [16]:
agent = CodeGeneratorAgent()

input_data = CodeGeneratorInput(
    modification_plan = result_data,
    user_query = user_question
)

result = await agent.generate_code_modifications(input_data)

INFO:agents.code_generator_agent.code_generator_agent:[CodeGeneratorAgent] Starting code generation for 1 files
INFO:agents.code_generator_agent.code_generator_agent:[CodeGeneratorAgent] Using direct replacement for examples/processing/feature_engineering.py
INFO:agents.code_generator_agent.code_generator_agent:[CodeGeneratorAgent] Using LLM-based modification for examples/processing/feature_engineering.py


Modification type ----------------- data_transformation


ERROR:agents.code_generator_agent.code_generator_agent:[CodeGeneratorAgent] LLM call failed for examples/processing/feature_engineering.py: Unknown service: ''. Valid service names are: accessanalyzer, account, acm, acm-pca, aiops, amp, amplify, amplifybackend, amplifyuibuilder, apigateway, apigatewaymanagementapi, apigatewayv2, appconfig, appconfigdata, appfabric, appflow, appintegrations, application-autoscaling, application-insights, application-signals, applicationcostprofiler, appmesh, apprunner, appstream, appsync, apptest, arc-region-switch, arc-zonal-shift, artifact, athena, auditmanager, autoscaling, autoscaling-plans, b2bi, backup, backup-gateway, backupsearch, batch, bcm-dashboards, bcm-data-exports, bcm-pricing-calculator, bcm-recommended-actions, bedrock, bedrock-agent, bedrock-agent-runtime, bedrock-agentcore, bedrock-agentcore-control, bedrock-data-automation, bedrock-data-automation-runtime, bedrock-runtime, billing, billingconductor, braket, budgets, ce, chatbot, chime